In [ ]:
!pip install -U transformers

In [ ]:
pip install -U "huggingface_hub"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 40.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import pandas as pd
from huggingface_hub import InferenceClient

from google.colab import userdata

hf_token = userdata.get("HF_token")

import time

import numpy as np
from sklearn.metrics import f1_score

from pathlib import Path

import random

import re

Mounted at /content/drive


In [ ]:
client = InferenceClient(api_key=hf_token)

In [ ]:
# Defining the function for sending a request to the LLM


def classify_text(text, model):
    prompt = f"""
    You are a political manifesto classifier.
    Your task is to rate quasi-sentences as 1 if it is about mental health policy or psychiatric care and 0 if it's not.

    Only output 1 or 0.

    This is the text to classify:
    {text}
    """
    prompt_with_format = f"""
You are a political manifesto classifier.
Your task is to rate quasi-sentences as 1 if it is about mental health policy or psychiatric care and 0 if it's not.

You should ALWAYS first give your reasoning using a <think> tag and your final result as a binary number in a <result> tag.

Example:

<text>
There are many cases of homelessness in our country.
</text>
<think>
Although can be mental health related reasons behind homelessness, this statement doesn't mention them explicitly.
</think>
<result>
0
</result>

Example:

<text>
Suicide is a leading cause of death amongst people aged between 25 and 50.
</text>
<think>
Suicides are closely associated with severe mental health issues.
</think>
<result>
1
</result>

Example:

<text>
We want to provide more support for families.
</text>
<think>
Text does not explicitly mention mental health support.
</think>
<result>
0
</result>

Example:

<text>
We will increase the funding for children and adolescent psychiatric treatment.
</text>
<think>
Paragraph does not explicitly mention mental health support.
</think>
<result>
1
</result>

Example:

<text>
Our party wants to increase the availability of counseling for asylum seekers.
</text>
<think>
Text does not spesify that it means psychological counseling - could also be e.g. career counseling.
</think>
<result>
0
</result>





---
<text>
{text}
</text>
<think>
"""

    try:
        # Using a fast, instruction-tuned model
        response = client.chat_completion(
            model=model,
            messages=[{"role": "user", "content": prompt_with_format}],
            max_tokens=200,  # To allow thinking mode
            temperature=0,  # Keep the outputs consistent
        )

        content = response.choices[0].message.content
        match = re.search(r"<result>\n(\d+)\n</result>", content)

        # breakpoint()
        if match:
            # Convert the extracted text directly to an integer
            result = int(match.group(1))
        else:
            result = -1
        # To see if something was not classified at all
        return result
    except Exception as e:
        print(f"Error classifying: {e}")
        return None

    finally:
        time.sleep(random.uniform(0.3, 1))

        # Adding a pause to avoid NAs

In [ ]:
# A list of models to try

my_models = {
    "Qwen": "Qwen/Qwen3-Coder-30B-A3B-Instruct",
    "Llama": "meta-llama/Llama-3.3-70B-Instruct",
    "Gemma": "google/gemma-3-27b-it",
    "ChatGPT": "openai/gpt-oss-20b",
    "Llama17B": "meta-llama/Llama-4-Scout-17B-16E-Instruct",
    "Qwen7B": "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Llama8B": "meta-llama/Llama-3.1-8B-Instruct",
    "Gemma12b": "google/gemma-3-12b-it",
    "Gemma4": "google/gemma-4-26B-A4B-it",
}

In [ ]:
# Reading the small test sample to python
file_path = "/content/drive/MyDrive/Colab Notebooks/Thesis/testing_sample_100.csv"
df = pd.read_csv(file_path)

In [ ]:
# Create a subset of the first 10 rows
test_df = df.head(10).copy()


# Run the classification on the subset only for a test that it works
print("Testing classification on the first 10 rows...")
test_df["predicted_score01"] = test_df["text"].apply(
    lambda text: classify_text(text, model=my_models["Llama"])
)

# View the results
print(test_df[["text", "score_manual", "predicted_score"]])

Testing classification on the first 10 rows...
                                                text  score_manual  \
0  We will increase resources for mental health c...             1   
1  Elaborate a protocol for the organization of S...             0   
2  The constant indebtedness of the state, even i...             0   
3  Increased participation plays a vital role in ...             1   
4  Ensure that all front-line public service prof...             1   
5  Development of a Knowledge Bank on Participati...             0   
6  the digital divide and communications, as well as             0   
7  We want to create more medical study places an...             1   
8  In other words, to avoid locating a public ser...             0   
9  The aim of Housing First is to integrate PSSAs...             1   

   predicted_score01  
0                  1  
1                  0  
2                  0  
3                  1  
4                  1  
5                  0  
6                  0 

In [ ]:
# Encoding the test set with LLama
df["predicted_score_Llama17B"] = df["text"].apply(
    lambda text: classify_text(text, model=my_models["Llama17B"])
)
# Changing the name here to try different models

# View the results
print(df[["text", "score_manual", "predicted_score_Llama17B"]])

# calculating the f1 accuracy score
f1_score(df.score_manual, df.predicted_score_Llama8B)

                                                 text  score_manual  \
0   We will increase resources for mental health c...             1   
1   Elaborate a protocol for the organization of S...             0   
2   The constant indebtedness of the state, even i...             0   
3   Increased participation plays a vital role in ...             1   
4   Ensure that all front-line public service prof...             1   
..                                                ...           ...   
95   introduce teaching of thoughtfully chosen su...             0   
96  Projects must demonstrate feasibility and adeq...             0   
97  Strong prevention and appropriate care - we wa...             1   
98  The introduction of terrestrial digital televi...             0   
99  That treatment for physical or mental health p...             1   

    predicted_score_Llama8B  
0                         1  
1                         0  
2                         0  
3                         0

ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].

In [ ]:
# Saving the results for further inspection

df.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Thesis/test01_classified_results.csv",
    index=False,
)

In [ ]:
# Testing on the large sample set

file_path = "/content/drive/MyDrive/Colab Notebooks/Thesis/testing_sample_500.csv"
df_500 = pd.read_csv(file_path)

In [ ]:
# Encoding the test set
df_500["predicted_score_Llama17B"] = df_500["text"].apply(
    lambda text: classify_text(text, model=my_models["Llama17B"])
)

# View the results
print(df_500[["text", "score_manual", "predicted_score_Llama17B"]])

# Calculating the f1 score
f1_score(df_500.score_manual, df_500.predicted_score_Llama17B)

                                                  text  score_manual  \
0                 The Copernican revolution is a fact.             0   
1    b) massive counseling on repatriation already ...             0   
2      We are experiencing an inflation of inequality.             0   
3          The right to work is part of a decent life.             0   
4    In addition, physical and mental health proble...             1   
..                                                 ...           ...   
495  We will increase salaries for academic staff a...             0   
496  In government, the SNP has appointed the UK’s ...             1   
497  Increase the range of supports to families and...             1   
498  Information campaigns coordinated with the Aut...             1   
499  The Liberal Party will pay special attention t...             1   

     predicted_score_Llama17B  
0                           0  
1                           0  
2                           0  
3      

0.992

In [ ]:
# Saving for closer inspection

df_500.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Thesis/llama17b_results.csv", index=False
)

In [ ]:
# testing on a whole manifesto

file_path_test = "/content/drive/MyDrive/Colab Notebooks/Thesis/11520_200609.csv"
df_test = pd.read_csv(file_path_test)

In [ ]:
# Apply to a test manifesto

print("Classifying rows... ")
df_test["predicted_score"] = df_test["text"].apply(
    lambda text: classify_text(text, model=my_models["Llama17B"])
)

Classifying rows... 


In [ ]:
# Checking results
print(df_test[["text", "predicted_score"]].head(15))

                                                 text  predicted_score
0            Parents are important for their children                0
1        and safe families are a strength for society                0
2                Children and families are different.                0
3   A stressful everyday life of managing family a...                0
4   Family policy should be characterized by freed...                0
5     , fairness between different forms of childcare                0
6   and provide the conditions for more time for c...                0
7   Childcare allowance should be introduced, maki...                0
8   Parental insurance should not be subject to qu...                0
9   Marriage, which is the most stable form of coh...                0
10  There should be a cap on the size of groups of...                0
11  There should also be pre-schools with differen...                0
12  Family centers, providing flexible and accessi...                0
13  Ma

In [ ]:
# Checking some rows that were classified 1
print(df_test[df_test["predicted_score"] == 1])

                                                 text  cmp_code  eu_code  pos  \
26               A patients' rights law is introduced       504      NaN   27   
29                                  offer care there.       504      NaN   30   
31              including by abolishing the Stop Law.       403      NaN   32   
32  A substantial investment in psychiatry is bein...       504      NaN   33   
76  The pupil should have the right to see a schoo...       504      NaN   77   

    manifesto_id  party    date language  annotations  translation_en  \
26  11520_200609  11520  200609  swedish         True            True   
29  11520_200609  11520  200609  swedish         True            True   
31  11520_200609  11520  200609  swedish         True            True   
32  11520_200609  11520  200609  swedish         True            True   
76  11520_200609  11520  200609  swedish         True            True   

    predicted_score  
26                1  
29                1  
31      

In [ ]:
# saving for closer inspection

df_test.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Thesis/manifesto_test.csv", index=False
)

In [ ]:
# Creating a summary df to save the results

summary_df = pd.DataFrame(
    columns=["manifesto_id", "language", "total_rows", "rows_classified_1"]
)

# creating a summary row of the just created classification

summary_row = pd.DataFrame(
    {
        "manifesto_id": [Path(file_path_test).stem],  # filename without .csv
        "orig_language": df_test["language"][2], # first row is title
        "total_rows": [len(df_test)],
        "rows_classified_1": [(df_test["predicted_score"] == 1).sum()],
    }
)

# checking it looks correct
print(summary_row)


# adding to the summary table
summary_df = pd.concat([summary_df, summary_row], ignore_index=True)

   manifesto_id orig_language  total_rows  rows_classified_1
0  11520_200609       swedish         124                  5


In [ ]:
# Testing on another manifesto

file_path_test = "/content/drive/MyDrive/Colab Notebooks/Thesis/22521_202103.csv"
df_test2 = pd.read_csv(file_path_test)

# Checking the length of the document
len(df_test2)

In [ ]:
# Encoding the second test manifesto

df_test2["predicted_score"] = df_test2["text"].apply(
    lambda text: classify_text(text, model=my_models["Llama17B"])
)

Error classifying: (Request ID: Root=1-6a9a94a3-7fdb4970057c2dbd37cf6f03;f856706c-632c-4330-aa49-8d496c99539b)

429 Too Many Requests for url: https://router.huggingface.co/v1/chat/completions.
Error classifying: (Request ID: Root=1-6a9a94a4-217f59281a52739028de2864;1d35e1a6-ce58-45c5-8df9-2716283c2ff1)

429 Too Many Requests for url: https://router.huggingface.co/v1/chat/completions.
Error classifying: (Request ID: Root=1-6a9a94e0-247fb7480397679850682e1b;5d0dce04-8eb2-4b0d-b9c4-954d99285206)

429 Too Many Requests for url: https://router.huggingface.co/v1/chat/completions.
Error classifying: (Request ID: Root=1-6a9a9524-7bef590056ae2f100b1bdf79;01e77fcb-5162-4fc5-97e6-09689f35f77b)

429 Too Many Requests for url: https://router.huggingface.co/v1/chat/completions.
Error classifying: (Request ID: Root=1-6a9a9525-498391921fe754084b945f14;5284b810-69e7-46c9-8749-d384485c8c15)

429 Too Many Requests for url: https://router.huggingface.co/v1/chat/completions.


KeyboardInterrupt: 

In [ ]:
# Taking a look at the results

print(df_test2[["text", "predicted_score"]].head(15))

print(df_test2[df_test2["predicted_score"] == 1])

KeyError: "['predicted_score'] not in index"

In [ ]:
# Saving for closer inspection

df_test2.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Thesis/manifesto_test2.csv", index=False
)